# 1D CNN-A — Index

This directory contains a **1D convolutional autoencoder** built in PyTorch, trained on
TSLA 1-minute OHLCV bars. The model learns a compressed representation of short-term
market behaviour without labels — purely from the shape of price, volume, and momentum.

Each notebook is self-contained: it runs the full data pipeline from CSV to clean windows,
then focuses on one specific stage of the learning path.  
All tuneable parameters live in [`config.py`](config.py).

---

## Notebooks

### Training

| Notebook | What it does |
|----------|--------------|
| [1dcnn_train.ipynb](1dcnn_train.ipynb) | Full data pipeline → engineer 14 features → scale → 64-bar sliding windows → gap filter → train 1D CNN autoencoder with `TrainingGuard` → save `model.pt` + `scaler.pkl` |

---

### Visualise Windows (before training — no model needed)

| Notebook | What it does |
|----------|--------------|
| [contact_sheet.ipynb](contact_sheet.ipynb) | Renders `N_SAMPLE` windows as a contact sheet — every 64 × 14-pixel window side by side at 4× scale |
| [heat_map.ipynb](heat_map.ipynb) | Renders `N_SAMPLE` windows as a heatmap strip — one row per window, 896 px wide, feature boundaries marked |
| [thumbnail_grid.ipynb](thumbnail_grid.ipynb) | Renders `N_SAMPLE` windows as a thumbnail grid — 10 × 10 px per window via Lanczos downsampling |

---

### Latent Space & Cluster Analysis (requires `model.pt`)

| Notebook | What it does |
|----------|--------------|
| [latent_cluster.ipynb](latent_cluster.ipynb) | Loads `model.pt`, extracts `LATENT_DIM`-dimensional latent vectors for every window, clusters with K-Means, visualises with t-SNE and centroid line plots → saves `kmeans.pkl` |
| [reconstruction.ipynb](reconstruction.ipynb) | Runs windows through encoder → decoder, compares original vs. reconstructed side by side, plots per-feature MSE and error distribution |
| [cluster_quality.ipynb](cluster_quality.ipynb) | Evaluates K = 2 … 16 using elbow inertia, silhouette score, and Davies-Bouldin index — helps confirm or update `N_CLUSTERS` in `config.py` |
| [temporal_patterns.ipynb](temporal_patterns.ipynb) | Plots cluster labels across the full timeline, by hour of day, and by day of week — reveals intraday and seasonal market regime structure |

---

### Walk-Forward Inference (requires `model.pt`, `scaler.pkl`, `kmeans.pkl`)

| Notebook | What it does |
|----------|--------------|
| [inference.ipynb](inference.ipynb) | Iterates bar-by-bar over a chosen date range; encodes each 64-bar window, measures reconstruction error (anomaly score), assigns cluster label, and displays a live-updating 5-panel dashboard |

---

## Suggested Run Order

```
1dcnn_train.ipynb          ← start here: trains the model; saves model.pt + scaler.pkl
    │
    ├── contact_sheet.ipynb       ┐
    ├── heat_map.ipynb            ├─ optional: explore raw windows before or after training
    └── thumbnail_grid.ipynb      ┘
    │
    └── latent_cluster.ipynb      ← cluster the latent space; saves kmeans.pkl
            │
            ├── reconstruction.ipynb    ← validate the model learned real structure
            ├── cluster_quality.ipynb   ← confirm N_CLUSTERS is the right number
            ├── temporal_patterns.ipynb ← discover when each regime appears in time
            └── inference.ipynb         ← walk-forward bar-by-bar; watch live MSE + cluster
```

The three visualisation notebooks (`contact_sheet`, `heat_map`, `thumbnail_grid`) are
independent — run them any time to inspect raw window data.  
Everything after `latent_cluster` requires `model.pt` to exist.  
`inference.ipynb` additionally requires `scaler.pkl` and `kmeans.pkl`.

---

## Configuration

All tuneable parameters are in [`config.py`](config.py) as a `@dataclass`:

| Group | Key parameters |
|-------|----------------|
| **Data** | `SYMBOL`, `TIMEFRAME`, `START_DATE`, `END_DATE`, `MAX_BARS` |
| **Windowing** | `WINDOW_SIZE`, `feature_cols` |
| **Autoencoder** | `LATENT_DIM`, `BATCH_SIZE`, `EPOCHS`, `LR`, `TEST_SPLIT` |
| **Training Guard** | `GUARD_PATIENCE`, `GUARD_MIN_DELTA`, `GUARD_OVERFIT_RATIO`, … |
| **Clustering** | `N_CLUSTERS`, `TSNE_SAMPLE`, `PLOT_FEATURES` |
| **Visualisation** | `N_SAMPLE`, `SCALE`, `GRID_COLS_A`, `THUMB_PX`, `GRID_COLS_C` |

Override defaults at the top of any notebook's Config cell:
```python
cfg = Config()
cfg.EPOCHS     = 30      # full training run
cfg.N_CLUSTERS = 12      # try more clusters
cfg.MAX_BARS   = None    # load all bars (~552k)
globals().update(vars(cfg))
```